In [ ]:
from dataclasses import asdict
from pathlib import Path
import os
import pandas as pd
from stock_etl import prepare_stock_prices, write_stock_csv, validate_stock_csv

# Work in this notebook's database/ directory. The input is read only.
# Set STOCK_SOURCE_XLSX to an existing source path instead of copying/editing the original.
source_excel = Path(os.environ.get('STOCK_SOURCE_XLSX', 'KSE_FIN_DATA_2023_database용.xlsx'))
if not source_excel.is_file():
    raise FileNotFoundError('Set STOCK_SOURCE_XLSX to the source workbook, or place a copy beside this notebook.')
# Use the price sheet by name, not its position among the financial sheets.
df = pd.read_excel(source_excel, sheet_name='수정주가')
eav_data_cleaned, report = prepare_stock_prices(df)
output_path = Path('stock_data_eav.csv')
write_stock_csv(eav_data_cleaned, output_path, overwrite=True)
validated_rows = validate_stock_csv(output_path)
print(asdict(report))
print(f'CSV validation passed: {len(validated_rows)} rows. This cell did not connect to or import into MySQL.')


In [ ]:
# 결과 출력: 삭제 후 NULL 값이 있는지 확인
eav_data_cleaned.head()

In [ ]:
asdict(report)


In [ ]:
import pandas as pd

# 파일을 읽기 (필요시 인코딩 설정)
df = pd.read_csv('fin_data.txt', delimiter='\t', encoding='utf-8')

# NULL 값 제거
df_cleaned = df.dropna()

# 전처리된 데이터를 새로운 파일로 저장
df_cleaned.to_csv('fin_data_cleaned.txt', sep='\t', index=False, encoding='utf-8')


In [ ]:
df_cleaned = pd.read_csv('fin_data_cleaned.txt', delimiter='\t', encoding='utf-8')
# 전처리: 특수 문자 제거 및 숫자 변환
columns_to_clean = ['총자산', '총부채', '총자본', '매출액', '영업이익', '당기순이익', 'EPS', 'BPS', 'SPS', 'CFPS', 'EBITDAPS']

# 각 열에서 괄호나 불필요한 문자 제거 후 숫자로 변환
for column in columns_to_clean:
    df_cleaned[column] = df_cleaned[column].replace({r'[^0-9.-]': ''}, regex=True).astype(float)


# 정제된 데이터를 새로운 파일로 저장
output_file_path_cleaned = 'fin_data_cleaned.txt'
df_cleaned.to_csv(output_file_path_cleaned, sep='\t', index=False, encoding='utf-8')
